In [22]:
import torch
print("Versi Torch:", torch.__version__)
print("Apakah CUDA (GPU) aktif?:", torch.cuda.is_available()) 
# Harusnya outputnya False (karena kita pakai CPU biar tidak error DLL)

from sentence_transformers import SentenceTransformer
print("Sentence Transformers berhasil di-load!")

Versi Torch: 2.9.1+cpu
Apakah CUDA (GPU) aktif?: False
Sentence Transformers berhasil di-load!


In [23]:

import PyPDF2
import os
import re
import numpy as np
import json
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
from sklearn.metrics.pairwise import cosine_similarity

print("✅ Library siap.")
PDF_PATH = "ALKITAB PDF-3-1162.pdf" # Pastikan file sudah diupload di folder yang sama

✅ Library siap.


In [24]:
# Tambahkan import ini
from sentence_transformers import CrossEncoder

# Load model khusus untuk menilai relevansi (Re-ranker)
# Model ini lebih teliti daripada model embedding biasa
print("Sedang mendownload model Re-Ranker...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("✅ Model Re-Ranker siap!")

Sedang mendownload model Re-Ranker...


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


✅ Model Re-Ranker siap!


In [25]:
def extract_text_from_pdf(pdf_path):
    text = ""
    if not os.path.exists(pdf_path):
        print(f"❌ File {pdf_path} tidak ditemukan!")
        return None
        
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in tqdm(reader.pages, desc="📖 Membaca PDF"):
            t = page.extract_text()
            if t:
                text += "\n" + t
    return text

print("✅ Fungsi ekstraksi siap.")

✅ Fungsi ekstraksi siap.


In [26]:
def parse_bible_complex(text):
    lines = [ln.strip() for ln in text.split("\n") if ln.strip()]
    
    data_items = [] 
    current_book = None
    current_pericope_title = "Umum"
    
    buffer_text = ""
    buffer_ref = ""
    
    # --- REGEX BARU ---
    # Menangkap format: (Kejadian 1:1-2:7) maupun (Kejadian 2:8-25)
    regex_pericope_ref = re.compile(r'^\(\s*(.+?)\s+(\d+:\d+(?:\s?[-–]\s?(?:\d+:\d+|\d+))?)\s*\)$')
    regex_verse_start = re.compile(r'^(\d+):(\d+)\s+(.*)')
    regex_chapter_only = re.compile(r'^\d+$')

    for i, line in enumerate(lines):
        # 1. Cek apakah ini Referensi Perikop?
        match_peri = regex_pericope_ref.match(line)
        if match_peri:
            # Simpan buffer ayat sebelumnya
            if buffer_ref and buffer_text:
                data_items.append({"type": "ayat", "ref": buffer_ref, "text": buffer_text.strip(), "parent": current_pericope_title})
                buffer_text = ""
                buffer_ref = ""

            # Ambil Baris SEBELUMNYA sebagai JUDUL PERIKOP
            if i > 0:
                judul_potensial = lines[i-1]
                if not regex_chapter_only.match(judul_potensial) and not regex_verse_start.match(judul_potensial):
                    current_pericope_title = judul_potensial
                    
                    kitab_name = match_peri.group(1).strip()
                    ref_range = match_peri.group(2).strip()
                    full_ref = f"{kitab_name} {ref_range}"
                    
                    # Simpan Perikop
                    data_items.append({
                        "type": "perikop", 
                        "ref": full_ref, 
                        "text": current_pericope_title, 
                        "parent": "-"
                    })
                    current_book = kitab_name
            continue

        # 2. Cek apakah ini Awal Ayat?
        match_verse = regex_verse_start.match(line)
        if match_verse:
            if buffer_ref and buffer_text:
                data_items.append({"type": "ayat", "ref": buffer_ref, "text": buffer_text.strip(), "parent": current_pericope_title})
            
            pasal = match_verse.group(1)
            ayat = match_verse.group(2)
            isi = match_verse.group(3)
            book_name = current_book if current_book else "Kitab"
            
            buffer_ref = f"{book_name} {pasal}:{ayat}"
            buffer_text = isi
            continue

        # 3. Abaikan angka bab saja
        if regex_chapter_only.match(line): continue

        # 4. Handle Text Wrapping
        is_next_line_ref = False
        if i + 1 < len(lines):
            if regex_pericope_ref.match(lines[i+1]): is_next_line_ref = True
        
        if buffer_ref and not is_next_line_ref:
            buffer_text += " " + line

    # Simpan sisa terakhir
    if buffer_ref and buffer_text:
        data_items.append({"type": "ayat", "ref": buffer_ref, "text": buffer_text.strip(), "parent": current_pericope_title})

    print(f"✅ Berhasil memparsing {len(data_items)} item (Ayat + Perikop).")
    return data_items

In [27]:
# 1. Baca PDF
raw_text = extract_text_from_pdf(PDF_PATH)

if raw_text:
    print(f"📄 Panjang teks: {len(raw_text):,} karakter")
    
    # 2. Lakukan Parsing
    dataset = parse_bible_complex(raw_text)
    
    # Intip 3 data pertama untuk memastikan benar
    print("\n🔍 Contoh 3 data awal:")
    print(json.dumps(dataset[:3], indent=2))
else:
    print("⚠️ PDF Kosong/Gagal dibaca.")

📖 Membaca PDF: 100%|██████████| 1160/1160 [00:37<00:00, 30.63it/s]


📄 Panjang teks: 5,231,737 karakter
✅ Berhasil memparsing 33414 item (Ayat + Perikop).

🔍 Contoh 3 data awal:
[
  {
    "type": "perikop",
    "ref": "Kejadian 1:1 -2:7",
    "text": "Allah menciptakan langit dan bumi serta isinya",
    "parent": "-"
  },
  {
    "type": "ayat",
    "ref": "Kejadian 1:1",
    "text": "Pada mulanya Allah menciptakan langit dan bumi.",
    "parent": "Allah menciptakan langit dan bumi serta isinya"
  },
  {
    "type": "ayat",
    "ref": "Kejadian 1:2",
    "text": "Bumi belum berbentuk dan kosong; gelap gulita menutupi samudera raya, dan Roh Allah melayang -layang di atas permukaan air.",
    "parent": "Allah menciptakan langit dan bumi serta isinya"
  }
]


In [28]:
def build_search_engine(data_items):
    print("⚙️ Memuat model S-BERT...")
    # Tokenisasi terjadi otomatis di dalam model.encode()
    model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    
    texts = [item['text'] for item in data_items]
    
    print("🧠 Membuat embedding (Vectorization)...")
    embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)
    embeddings = embeddings.astype('float32')

    # Buat Index FAISS
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    
    return model, index

# Jalankan fungsi
if 'dataset' in locals() and dataset:
    model, index = build_search_engine(dataset)
    print("✅ Indexing Selesai! Siap digunakan.")
else:
    print("⚠️ Dataset belum siap. Jalankan Cell 4 dulu.")

⚙️ Memuat model S-BERT...
🧠 Membuat embedding (Vectorization)...


Batches: 100%|██████████| 523/523 [03:29<00:00,  2.50it/s]

✅ Indexing Selesai! Siap digunakan.


In [29]:
def search_bible_rerank(query, model, index, data_items, cross_encoder, top_k=5):
    print(f"\n🔎 Query: '{query}'")
    
    # 1. TAHAP 1: PENCARIAN KASAR (Retrieve)
    # Ambil 50 kandidat teratas dulu (biar kita punya banyak pilihan untuk disaring)
    query_vec = model.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, 50) 
    
    # Siapkan data untuk Re-Ranking
    candidates = []
    candidate_indices = []
    
    for idx in indices[0]:
        if idx < len(data_items):
            item_text = data_items[idx]['text']
            candidates.append([query, item_text]) # Pasangkan Query dengan Ayat
            candidate_indices.append(idx)
            
    # 2. TAHAP 2: PENILAIAN ULANG (Re-Ranking)
    # Cross-Encoder akan menilai seberapa cocok pasangan (Query - Ayat)
    if not candidates:
        print("Tidak ada data ditemukan.")
        return

    scores = cross_encoder.predict(candidates)
    
    # Gabungkan index asli dengan skor baru, lalu urutkan dari skor tertinggi
    scored_results = sorted(list(zip(candidate_indices, scores)), key=lambda x: x[1], reverse=True)
    
    # 3. TAMPILKAN HASIL TERBAIK
    results_perikop = []
    results_ayat = []
    seen_refs = set()

    for idx, score in scored_results:
        item = data_items[idx]
        
        # Filter duplikasi
        if item['ref'] in seen_refs: continue
        seen_refs.add(item['ref'])
        
        if item['type'] == 'perikop':
            results_perikop.append(item)
        else:
            results_ayat.append(item)

    # --- OUTPUT ---
    print("\n📂 TEMA / JUDUL PERIKOP (Re-Ranked):")
    for p in results_perikop[:5]: # Ambil 5 teratas
        print(f" 🔹 {p['text']}")
        print(f"    👉 Ref: {p['ref']}")

    print(f"\n📖 AYAT RELEVAN (Top {top_k}):")
    count = 0
    for res in results_ayat:
        count += 1
        print(f" {count}. [{res['ref']}]")
        print(f"    {res['text']}")
        print(f"    🏷️ Konteks: {res['parent']}\n")
        if count >= top_k: break

print("✅ Fungsi search_bible_rerank siap digunakan.")

✅ Fungsi search_bible_rerank siap digunakan.


In [38]:
# Ganti kata kunci di sini
KEYWORD = "tuhan menciptakan" 

search_bible_rerank(KEYWORD, model, index, dataset, cross_encoder)


🔎 Query: 'tuhan menciptakan'

📂 TEMA / JUDUL PERIKOP (Re-Ranked):
 🔹 TUHAN adalah Pencipta
    👉 Ref: Yesaya 45:9 -19
 🔹 TUHAN membangkitkan seorang pembebas
    👉 Ref: Yesaya 41:1 -7
 🔹 Allah menciptakan langit dan bumi serta isinya
    👉 Ref: Kejadian 1:1 -2:7
 🔹 Kebesaran TUHAN dalam segala ciptaan- Nya
    👉 Ref: Mazmur 104:1 -35
 🔹 Kedatangan Tuhan
    👉 Ref: 1 Tesalonika 4:13 -18

📖 AYAT RELEVAN (Top 5):
 1. [Amsal 8:22]
    TUHAN telah menciptakan aku sebagai permulaan pekerjaan -Nya, sebagai perbuatan -Nya yang pertama- tama dahulu kala.
    🏷️ Konteks: Wejangan hikmat

 2. [Yesaya 45:18]
    Sebab beginilah firman TUHAN, yang menciptakan langit, --Dialah Allah --yang membentuk bumi dan menjadikannya dan yang menegakkannya, --dan Ia menciptakannya bukan supaya kosong, tetapi Ia membentuknya untuk didiami-- :"Akulah TUHAN dan tidak ada yang lain.
    🏷️ Konteks: TUHAN adalah Pencipta

 3. [Yesaya 40:28]
    Tidakkah kautahu, dan tidakkah kaudengar? TUHAN ialah Allah kekal yang 

In [31]:
def evaluate_query(query, top_k=5):
    query_vec = model.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, top_k)
    
    # Ambil item hasil
    results = [dataset[i] for i in indices[0] if i < len(dataset)]
    
    # Hitung cosine similarity manual untuk display
    res_vecs = model.encode([r['text'] for r in results])
    scores = cosine_similarity(query_vec, res_vecs)[0]
    
    print(f"📊 Evaluasi Skor untuk: '{query}'")
    print("-" * 50)
    for i, (score, item) in enumerate(zip(scores, results), 1):
        jenis = "TEMA" if item['type'] == 'perikop' else "AYAT"
        print(f"{i}. [{jenis}] {item['ref']} (Score: {score:.4f})")
        print(f"   {item['text'][:100]}...")
    print("-" * 50)
    print(f"💞 Rata-rata Score: {np.mean(scores):.4f}")

# Coba Evaluasi
evaluate_query("Manusia jatuh dalam dosa")

📊 Evaluasi Skor untuk: 'Manusia jatuh dalam dosa'
--------------------------------------------------
1. [TEMA] Kejadian 3:1 -24 (Score: 0.9960)
   Manusia jatuh ke dalam dosa...
2. [TEMA] Mazmur 51:1 -21 (Score: 0.8568)
   Pengakuan dosa...
3. [TEMA] Imamat 4:1 -5:13 (Score: 0.8269)
   Korban penghapus dosa...
4. [TEMA] Imamat 6:24 -30 (Score: 0.8269)
   Korban penghapus dosa...
5. [AYAT] 2 Korintus 5:21 (Score: 0.7711)
   Dia yang tidak mengenal dosa telah dibuat -Nya menjadi dosa karena kita, supaya dalam Dia kita diben...
--------------------------------------------------
💞 Rata-rata Score: 0.8555
